# LC 23 — Merge K Sorted Lists

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Maintain a min-heap of size at
most k, each entry holding the current head of each list. Always
extract the global minimum in O(log k), then push the next node
from that list. This avoids scanning all k list heads each time.
</div>

## Official Problem Statement

Given an array of `k` linked-lists, each sorted in ascending
order, merge all linked-lists into one sorted linked-list and
return it.

**Example 1:**
```
Input:
  lists = [[1,4,5],[1,3,4],[2,6]]
Output: [1,1,2,3,4,4,5,6]
```
**Example 2:**
```
Input:  lists = []
Output: []
```
**Constraints:**
- `k == len(lists)`
- `0 <= k <= 10^4`
- `0 <= lists[i].length <= 500`
- `-10^4 <= lists[i][j] <= 10^4`
- Each list is sorted in non-decreasing order.
- Total nodes: `<= 10^4`

## What This Is Actually Asking

Merging 2 sorted lists is easy (O(n)). For k lists, a naive
approach scans all k current heads to find the minimum —
O(k) per node, O(nk) total.

The heap approach keeps only the current frontier (one node
per list) in a min-heap. Extract-min is O(log k), and we do
it once per node, giving O(n log k) total.

One pitfall: Python's heap comparison falls through to the node
object if values tie — we store `(val, list_idx, node)` so
comparison never reaches the node (which isn't comparable).

## Walk Through an Example by Hand

```
lists = [[1,4,5], [1,3,4], [2,6]]
         list 0    list 1   list 2

Initial heap (val, list_idx, node):
  [(1,0,Node1), (1,1,Node1), (2,2,Node2)]
  heap top = (1, 0, Node(1))

Pop (1,0): append 1 to result. Node(1).next=Node(4)
  push (4, 0, Node(4))
  heap: [(1,1,N1),(2,2,N2),(4,0,N4)]

Pop (1,1): append 1. Node(1).next=Node(3)
  push (3, 1, Node(3))
  heap: [(2,2,N2),(3,1,N3),(4,0,N4)]

Pop (2,2): append 2. Node(2).next=Node(6)
  push (6, 2, Node(6))
  heap: [(3,1,N3),(4,0,N4),(6,2,N6)]

Pop (3,1): append 3. Node(3).next=Node(4)
Pop (4,0): append 4. Node(4).next=Node(5)
Pop (4,1): append 4. Node(4).next=None  (no push)
Pop (5,0): append 5. Node(5).next=None
Pop (6,2): append 6. Node(6).next=None

Result: 1→1→2→3→4→4→5→6 ✓
```

## The Picture

```
k lists, each sorted:

  List 0:  1 → 4 → 5
  List 1:  1 → 3 → 4
  List 2:  2 → 6

Min-heap holds the CURRENT HEAD of each list:

  ┌─────────────────────────────┐
  │  MIN-HEAP (size ≤ k)        │
  │  (val, list_idx, node)      │
  │  (1,0,•)  (1,1,•)  (2,2,•) │
  │      ↑                      │
  │   top = global min          │
  └─────────────────────────────┘
           ↓ pop
  append to result, push next node from same list

  Each pop + push: O(log k)
  Total nodes: n  →  Total: O(n log k)

  Tiebreaker: (val, list_idx, node)
  → list_idx breaks value ties so node is never compared
```

## When To Use This Pattern

- When you need the **global minimum** from k sorted sources
  repeatedly, think **min-heap of size k**.
- When merging k sorted streams (lists, iterators, files),
  think **heap-based k-way merge**.
- When heap comparison might fail on objects, think
  **add a tiebreaker (index)** to the tuple.
- When k is large but each extraction is O(log k), think
  **heap over naive O(k) linear scan**.
- When the output must be built node-by-node, think
  **dummy head linked list** for clean pointer management.

## The Approach

Create a dummy head node. Push `(val, list_idx, node)` for the
first node of every non-empty list onto a min-heap. Repeatedly
pop the minimum, append it to the result list by pointer, and
if the popped node has a next, push `(next.val, list_idx, next)`
onto the heap.

Continue until the heap is empty. Return `dummy.next`. The
`list_idx` tiebreaker ensures no two tuples compare their node
objects, avoiding a TypeError.

In [ ]:
# Imports + ListNode helpers
import heapq
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    d = ListNode(0); c = d
    for v in vals:
        c.next = ListNode(v); c = c.next
    return d.next

def to_list(h):
    r = []
    while h: r.append(h.val); h = h.next
    return r

In [ ]:
# ----------------------------------------------------------
# Harness
# ----------------------------------------------------------
def test_harness(func):
    cases = [
        (
            [[1,4,5],[1,3,4],[2,6]],
            [1,1,2,3,4,4,5,6]
        ),
        ([], []),
        ([[]], []),
        ([[1],[0]], [0,1]),
        ([[2],[],[1,3]], [1,2,3]),
    ]
    passed = 0
    for raw_lists, expected in cases:
        lists = [make_list(l) for l in raw_lists]
        result = to_list(func(lists))
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | input={raw_lists}"
                  f" | expected={expected}"
                  f" | got={result}")
        else:
            print(f"{status} | {raw_lists}"
                  f" → {result}")
        passed += ok
    print(f"\n{passed}/{len(cases)} tests passed")

In [ ]:
def mergeKLists(
    lists: List[Optional[ListNode]]
) -> Optional[ListNode]:
    """
    LC 23 — Merge K Sorted Lists

    Min-heap of (val, list_idx, node):
    1. Push first node of each non-empty list
    2. Pop min, append to result via dummy head
    3. If popped node has next: push (next.val,
       list_idx, next)
    4. Return dummy.next

    Tiebreaker: list_idx prevents comparing ListNode
    objects (which have no __lt__)

    Time:  O(n log k) n=total nodes, k=num lists
    Space: O(k) for the heap
    """
    pass
    # Debug hints:
    # print(f"popped val={val} list={idx}")
    # print(f"heap size={len(heap)}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(mergeKLists)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Collect all + sort | O(n log n) | O(n) | Simple but loses sorted structure |
| Sequential merge 2 at a time | O(nk) | O(1) | Slow for large k |
| Divide & conquer merges | O(n log k) | O(log k) | Stack space for recursion |
| Min-heap (optimal) | O(n log k) | O(k) | Industry standard |

## Real World Connection

**AWS / DE context:** Merging k sorted runs is the core of
external merge sort — used whenever data doesn't fit in memory.
AWS Glue's shuffle phase and Spark's sort-merge join both use
k-way merge on sorted partitions, with heap-based merging
at the reducer stage.

At Citi, end-of-day trade reconciliation merges sorted position
files from k regional systems. The heap merge pattern runs in
O(n log k) regardless of how many source systems exist.

In streaming data pipelines (Kinesis, Kafka), merging k
partitioned sorted streams (by timestamp or key) into one
ordered output is exactly this problem — the heap ensures
global ordering with minimal memory.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra